# Reading and Writing Data

**Learning Objectives:**
- Master `read_csv()` with key parameters
- Read Excel, JSON, SQL, and HTML data
- Write DataFrames to CSV, Excel, JSON, and Parquet
- Handle large CSV files with chunking
- Understand fast I/O with Parquet

## 1. Setup – Sample Data

We'll create in-memory file-like objects using `StringIO` / `BytesIO`.

In [ ]:
import pandas as pd
import numpy as np
from io import StringIO, BytesIO

# Sample CSV content
csv_data = """ID,Name,Salary,Department,Hire_Date
1,Alice,75000,IT,2022-06-15
2,Bob,68000,HR,2023-01-20
3,Charlie,82000,IT,2021-03-10
4,Diana,72000,Finance,2022-11-01
5,Eve,65000,HR,2024-02-28
"""
print("CSV data ready.")

## 2. `read_csv()` – Key Parameters

Pandas `read_csv()` is highly configurable.

In [ ]:
# Basic read
df = pd.read_csv(StringIO(csv_data))
print("Basic read:")
print(df)

# Using custom separator, no header
tsv_data = "1\tAlice\t75000\n2\tBob\t68000\n"
df_tsv = pd.read_csv(StringIO(tsv_data), sep="\t", header=None,
                      names=["ID", "Name", "Salary"])
print("\nCustom sep + no header:")
print(df_tsv)

In [ ]:
# index_col, usecols, parse_dates, na_values
df_advanced = pd.read_csv(
    StringIO(csv_data),
    index_col="ID",
    usecols=["ID", "Name", "Salary", "Hire_Date"],
    parse_dates=["Hire_Date"],
    na_values={"Salary": ["NA", "NULL"]}
)
print("Advanced read:")
print(df_advanced)
print("\nHire_Date dtype:", df_advanced["Hire_Date"].dtype)

In [ ]:
# dtype specification, skiprows, nrows
df_dtypes = pd.read_csv(
    StringIO(csv_data),
    dtype={"ID": "int8", "Salary": "float32"},
    skiprows=[2],  # skip row with Bob (0-indexed, skip index 2 => row 3)
    nrows=3
)
print("With dtype + skiprows + nrows:")
print(df_dtypes)
print("Dtypes:", df_dtypes.dtypes.to_dict())

## 3. `read_excel()` – Excel Files

Read Excel workbooks (requires `openpyxl` or `xlrd`).

In [ ]:
# Write a DataFrame to Excel, then read it back
df.to_excel("temp_sample.xlsx", sheet_name="Employees", index=False)
df_excel = pd.read_excel("temp_sample.xlsx", sheet_name="Employees")
print("Read from Excel:")
print(df_excel)

# Clean up
import os
os.remove("temp_sample.xlsx")

## 4. `read_json()` – JSON Data

In [ ]:
json_data = """[
  {"ID": 1, "Name": "Alice", "Salary": 75000},
  {"ID": 2, "Name": "Bob", "Salary": 68000}
]
"""
df_json = pd.read_json(StringIO(json_data))
print("Read from JSON:")
print(df_json)

## 5. `read_sql()` – SQL Databases

Execute SQL queries and load results into a DataFrame.

In [ ]:
import sqlite3

# Create an in-memory SQLite database
conn = sqlite3.connect(":memory:")
df.to_sql("employees", conn, index=False, if_exists="replace")

# Read via SQL query
df_sql = pd.read_sql("SELECT Name, Salary FROM employees WHERE Salary > 70000", conn)
print("From SQL:")
print(df_sql)
conn.close()

## 6. `read_html()` – HTML Tables

Extract tables from HTML pages.

In [ ]:
html_data = """<table>
  <tr><th>Name</th><th>Score</th></tr>
  <tr><td>Alice</td><td>95</td></tr>
  <tr><td>Bob</td><td>87</td></tr>
</table>
"""
dfs_html = pd.read_html(html_data)
print("From HTML (returns list of DataFrames):")
print(dfs_html[0])

## 7. Writing Data

`to_csv()`, `to_excel()`, `to_json()`, and `to_parquet()`.

In [ ]:
# to_csv with options
out_csv = StringIO()
df.to_csv(out_csv, index=False, sep=";", float_format="%.2f")
print("CSV output:")
print(out_csv.getvalue())

# to_json
print("\nJSON output:")
print(df.to_json(orient="records", indent=2))

## 8. Reading Large CSVs in Chunks

Use `chunksize` to avoid loading all data into memory at once.

In [ ]:
# Simulate a large CSV by writing and re-reading in chunks
large_csv = StringIO()
pd.DataFrame({"x": range(100), "y": np.random.randn(100)}).to_csv(large_csv, index=False)
large_csv.seek(0)

# Read in chunks of 30 rows
chunks = pd.read_csv(large_csv, chunksize=30)
total = 0
for i, chunk in enumerate(chunks):
    total += chunk["y"].sum()
    print(f"Chunk {i+1}: rows {chunk.index[0]}-{chunk.index[-1]}, partial sum={chunk['y'].sum():.2f}")
print(f"\nTotal sum across all chunks: {total:.2f}")

## 9. Parquet – Fast Columnar I/O

Parquet is a compressed, columnar format that's much faster than CSV for large data.

In [ ]:
# Requires pyarrow or fastparquet
try:
    df.to_parquet("temp_sample.parquet", index=False)
    df_pq = pd.read_parquet("temp_sample.parquet")
    print("Parquet round-trip:")
    print(df_pq)
    import os
    os.remove("temp_sample.parquet")
except Exception as e:
    print("Parquet not available:", e)
    print("Install with: pip install pyarrow")

## 10. Practice Problems

In [ ]:
# Problem 1: Read a CSV with custom parameters:
#            - separator is '|'
#            - no header, columns = ['A','B','C']
#            Only keep rows where column A > 50.
pipe_csv = "ID|Name|Score\n1|Alice|85\n2|Bob|92\n3|Charlie|78\n"
df_pipe = pd.read_csv(StringIO(pipe_csv), sep="|")
print("Pipe-delimited CSV:")
print(df_pipe)

In [ ]:
# Problem 2: Export df to JSON (orient='records'), then read it back.
import json
# Export
json_out = df.to_json(orient="records")
# Import
df_roundtrip = pd.read_json(StringIO(json_out))
print("Round-trip through JSON:")
print(df_roundtrip)

In [ ]:
# Problem 3: Write a loop that reads a CSV in chunks of size 10
#            and computes the running average of a column.
large_csv2 = StringIO()
pd.DataFrame({"val": np.random.randn(50).cumsum()}).to_csv(large_csv2, index=False)
large_csv2.seek(0)

running_sum = 0
running_count = 0
for chunk in pd.read_csv(large_csv2, chunksize=10):
    running_sum += chunk["val"].sum()
    running_count += len(chunk)
    print(f"Running average after {running_count} rows: {running_sum / running_count:.3f}")

## 11. Summary

| Function | Format | Key Parameters |
|----------|--------|---------------|
| `read_csv()` | CSV / TSV | `sep`, `header`, `names`, `index_col`, `usecols`, `dtype`, `parse_dates`, `na_values`, `skiprows`, `nrows`, `chunksize` |
| `read_excel()` | Excel | `sheet_name`, `header`, `index_col` |
| `read_json()` | JSON | `orient` |
| `read_sql()` | SQL | `sql`, `con`, `params` |
| `read_html()` | HTML | `match`, `attrs` |
| `to_csv()` / `to_excel()` / `to_json()` / `to_parquet()` | Write formats | `index`, `columns`, `compression` |